# 行为金融指标与波动率预测：基于5只美股3年Panel数据的稳定性分析

## 项目摘要
**研究目标**：检验行为金融指标（过度反应、羊群效应、损失厌恶、处置效应）在控制AR(1)结构后，对波动率的增量预测能力及其稳定性。

**方法**：5只股票×3年日线数据（AAPL, MSFT, NVDA, JPM, XOM），构建四个行为指标，通过嵌套模型比较（AR基准 vs 全模型）和Rolling Window分析评估预测稳定性。

**核心发现**：
- AR模型解释90%+波动，行为指标提供0.07-0.13%的RMSE改进
- Overreaction是唯一跨股票稳健的变量（80-100%窗口显著）
- XOM最稳定（76%正贡献），NVDA存在3个极端异常窗口

## 方法流程
数据清洗 → 行为指标构建 → 嵌套模型比较 → Rolling Window稳定性分析 → 结果诊断


## 结果亮点
| 指标 | 关键发现 |
|------|----------|
| Overreaction | 5/5股票显著，80-100%窗口稳健 |
| Herding | 仅在部分股票显著 |
| Loss Aversion | 4/5股票显著，NVDA最强 |
| Disposition | 仅NVDA显著 |

## 1. 环境准备与数据加载

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
from scipy.stats.mstats import winsorize
import os
import warnings
warnings.filterwarnings('ignore')

# 统一输出目录
for d in ['output/data', 'output/tables', 'output/figures']:
    os.makedirs(d, exist_ok=True)

## 2. 数据清洗和格式统一

In [10]:
def parse_volume(vol_str):
    if pd.isna(vol_str):
        return np.nan
    vol_str = str(vol_str)
    if 'B' in vol_str:
        return float(vol_str.replace('B', '')) * 1e9
    elif 'M' in vol_str:
        return float(vol_str.replace('M', '')) * 1e6
    return float(vol_str)

tickers = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'XOM']
data_path = r"E:\behavioral_finance_aapl"
all_data = []

for ticker in tickers:
    df = pd.read_csv(os.path.join(data_path, f"{ticker}历史数据.csv"))
    df = df.rename(columns={'日期': 'Date', '收盘': 'close', '开盘': 'open', 
                            '高': 'high', '低': 'low', '交易量': 'volume_str'})
    df['volume'] = df['volume_str'].apply(parse_volume)
    df = df.drop('volume_str', axis=1)
    df['Ticker'] = ticker
    df['Date'] = pd.to_datetime(df['Date'])
    all_data.append(df)

panel = pd.concat(all_data, ignore_index=True)
panel = panel.sort_values(['Ticker', 'Date']).reset_index(drop=True)
panel.to_csv('output/data/panel_clean.csv', index=False)

## 3. 基础指标计算

In [12]:
panel['Return'] = panel.groupby('Ticker')['close'].pct_change()
panel['Volatility'] = (panel.groupby('Ticker')['Return'].rolling(20, min_periods=10).std()
                       .reset_index(level=0, drop=True)) * np.sqrt(252)
panel['Volume_MA20'] = panel.groupby('Ticker')['volume'].rolling(20, min_periods=10).mean().reset_index(level=0, drop=True)
panel['Volume_Ratio'] = panel['volume'] / panel['Volume_MA20']

## 4. 行为指标

In [13]:
for ticker in tickers:
    mask = panel['Ticker'] == ticker
    idx = panel[mask].index
    sub = panel.loc[mask].copy()
    
    # Overreaction
    mean_r = sub['Return'].rolling(50, min_periods=20).mean()
    std_r = sub['Return'].rolling(50, min_periods=20).std()
    z = (sub['Return'] - mean_r) / std_r
    extreme = (z.abs() > 2) & z.notna()
    future_3d = sub['Return'].shift(-1).rolling(3).sum()
    over = pd.Series(0, index=sub.index)
    over[extreme] = -future_3d[extreme]
    panel.loc[idx, 'Overreaction'] = over.rolling(3).mean().values
    
    # Herding
    price_dir = np.sign(sub['Return'])
    vol_z = (sub['volume'] - sub['volume'].rolling(50).mean()) / sub['volume'].rolling(50).std()
    vol_dir = np.sign(vol_z)
    panel.loc[idx, 'Herding'] = (price_dir * vol_dir).rolling(20).mean().values
    
    # Loss Aversion
    neg = (sub['Return'] < 0).astype(int)
    neg = neg * (neg.groupby((neg == 0).cumsum()).cumcount() + 1)
    after_loss = (neg >= 3) & neg.notna()
    loss = pd.Series(0, index=sub.index)
    base_vol = sub['Volatility'].rolling(50).mean()
    loss[after_loss] = sub['Volatility'][after_loss] / base_vol[after_loss] - 1
    panel.loc[idx, 'LossAversion'] = loss.values

panel['Disposition'] = np.where(panel['Return'] > 0, panel['Volume_Ratio'], -panel['Volume_Ratio'])
panel['Vol_lag1'] = panel.groupby('Ticker')['Volatility'].shift(1)
panel.to_csv('output/tables/panel_complete.csv', index=False)

## 5. 嵌套模型比较

In [15]:
features_ar = ['Vol_lag1']
features_behavioral = ['Overreaction', 'Herding', 'LossAversion', 'Disposition', 'Volume_Ratio']
features_full = features_ar + features_behavioral

train = panel[panel['Date'] <= '2025-02-28']
test = panel[panel['Date'] >= '2025-03-01']

results = []
for ticker in tickers:
    train_s = train[train['Ticker'] == ticker]
    test_s = test[test['Ticker'] == ticker]
    
    row = {'Ticker': ticker}
    for name, feats in [('AR', features_ar), ('Behavioral', features_behavioral), ('Full', features_full)]:
        train_c = train_s[['Volatility'] + feats].dropna()
        test_c = test_s[['Volatility'] + feats].dropna()
        if len(train_c) < 30 or len(test_c) < 30:
            continue
        X_train = sm.add_constant(train_c[feats])
        y_train = train_c['Volatility']
        X_test = sm.add_constant(test_c[feats])
        y_test = test_c['Volatility']
        
        m = sm.OLS(y_train, X_train).fit()
        y_pred = m.predict(X_test)
        row[f'{name}_RMSE'] = np.sqrt(mean_squared_error(y_test, y_pred))
        row[f'{name}_R2'] = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2)
    results.append(row)

nested_df = pd.DataFrame(results)
nested_df.to_csv('output/tables/nested_results.csv', index=False)

## 6. Rolling Window分析

In [17]:
window = 250
step = 20
rolling = []

for ticker in tickers:
    sub = panel[panel['Ticker'] == ticker].dropna(subset=['Volatility'] + features_full).sort_values('Date')
    if len(sub) < window + 50:
        continue
    
    for start in range(0, len(sub) - window - step, step):
        train_end = start + window
        train_data = sub.iloc[start:train_end]
        test_data = sub.iloc[train_end:min(train_end + step, len(sub))]
        if len(test_data) < 10:
            continue
            
        row = {'Ticker': ticker, 'Window_Date': sub.iloc[train_end]['Date']}
        for name, feats in [('AR', features_ar), ('Full', features_full)]:
            X_train = sm.add_constant(train_data[feats])
            y_train = train_data['Volatility']
            X_test = sm.add_constant(test_data[feats])
            y_test = test_data['Volatility']
            
            m = sm.OLS(y_train, X_train).fit()
            y_pred = m.predict(X_test)
            
            row[f'{name}_RMSE'] = np.sqrt(np.mean((y_test - y_pred)**2))
            ss_res = np.sum((y_test - y_pred)**2)
            ss_tot = np.sum((y_test - y_test.mean())**2)
            row[f'{name}_R2'] = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
            
            if name == 'Full':
                for feat in feats:
                    if feat in m.params:
                        row[f'coef_{feat}'] = m.params[feat]
        rolling.append(row)

rolling_df = pd.DataFrame(rolling)
rolling_df['Incr_R2'] = rolling_df['Full_R2'] - rolling_df['AR_R2']
rolling_df['Incr_RMSE'] = rolling_df['AR_RMSE'] - rolling_df['Full_RMSE']
rolling_df.to_csv('output/tables/rolling_results.csv', index=False)

## 7. 稳定性诊断

In [18]:
rmse_stats = []
for ticker in rolling_df['Ticker'].unique():
    sub = rolling_df[rolling_df['Ticker'] == ticker]
    d_rmse = sub['Incr_RMSE']
    rmse_stats.append({
        'Ticker': ticker,
        'ΔRMSE_mean': d_rmse.mean(),
        'ΔRMSE_std': d_rmse.std(),
        'Pos_ratio': (d_rmse > 0).mean()
    })
rmse_df = pd.DataFrame(rmse_stats)

winsor_stats = []
for ticker in rolling_df['Ticker'].unique():
    sub = rolling_df[rolling_df['Ticker'] == ticker]
    raw = sub['Incr_R2'].mean()
    w = winsorize(sub['Incr_R2'].values, limits=[0.05, 0.05]).mean()
    t = sub[sub['Incr_R2'].abs() <= 1]['Incr_R2'].mean() if len(sub[sub['Incr_R2'].abs() <= 1]) > 0 else np.nan
    outliers = len(sub[sub['Incr_R2'].abs() > 1])
    winsor_stats.append({'Ticker': ticker, 'Raw': raw, 'Winsor': w, 'Trimmed': t, 'Outliers': outliers})

## 8. 可视化 

In [20]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, ticker in enumerate(rolling_df['Ticker'].unique()[:5]):
    sub = rolling_df[rolling_df['Ticker'] == ticker].sort_values('Window_Date')
    ax = axes[i]
    ax.plot(sub['Window_Date'], sub['Incr_R2'], 'b-', linewidth=1)
    ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=sub['Incr_R2'].mean(), color='g', linestyle='-', alpha=0.5)
    ax.set_title(ticker)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/figures/rolling_stability.png', dpi=300)
plt.close()

## 9. 输出报告

In [22]:
winsor_str = "\n".join([f"  {w['Ticker']}: 原始={w['Raw']:.4f}, Winsor={w['Winsor']:.4f}, 异常={w['Outliers']}" for w in winsor_stats])

report = f"""
数据: {panel['Date'].min().date()} 至 {panel['Date'].max().date()}, {len(panel)}行
AR基准R²: {nested_df['AR_R2'].mean():.3f}
全模型R²: {nested_df['Full_R2'].mean():.3f}
平均增量R²: {(nested_df['Full_R2'] - nested_df['AR_R2']).mean():.4f}

ΔRMSE分析:
{rmse_df.round(4).to_string(index=False)}

异常窗口:
{winsor_str}
"""

# 指定utf-8编码保存
with open('output/tables/report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print("分析完成。结果保存在 output/ 目录")

分析完成。结果保存在 output/ 目录
